# `ref_spectra` tutorial

This notebook shows how to work with the reference spectra in `empylib.ref_spectra`: solar and thermal sources, atmospheric transmission, photopic weighting, spectral averages, and plotting helpers.

**Learning goals**

- load packaged reference spectra and compare them on a common wavelength grid
- compute solar and thermal averages of a spectral property
- plot spectral data with the library helper instead of hand-building every figure

**Notebook design**

- every runnable cell calls the public `empylib` API directly
- parameter meanings are explained in markdown and in short inline comments
- outputs are inspected in the same notebook so you can see what each function returns
- the core path is offline-first; internet-backed examples live in clearly marked optional appendices

In [ ]:
from pathlib import Path
import os
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "empylib").exists() and (candidate / "docs").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the EMPI Lib repository root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.figsize"] = (7, 3)

import empylib.ref_spectra as ref

## Load reference spectra and compare the main sources

**Functions used**

- ref.read_spectrafile
- ref.AM15
- ref.T_atmosphere
- ref.T_atmosphere_hemi
- ref.Bplanck

**Problem we are solving**

Many optical calculations need a weighting spectrum or an atmospheric transmission curve. This section shows how to fetch the most common packaged references on a shared wavelength grid.

**Parameter guide for this example**

- `lam`: wavelength grid in micrometers used for all interpolated spectra
- `MaterialName='T_atmosphere.txt'`: packaged text file loaded with `read_spectrafile`
- `spectra_type`: choose `'global'` or `'direct'` for AM1.5
- `beta_tilt`: collector tilt angle in degrees for hemispherical atmosphere
- `T`: blackbody temperature in Kelvin for `Bplanck`

**Outputs to inspect**

- interpolated arrays for AM1.5, atmospheric transmission, and Planck emission
- `atm_table`: source table used by the atmospheric file reader

In [ ]:
lam = np.linspace(0.30, 25.0, 400)

atm_values, atm_table = ref.read_spectrafile(
    lam,
    "T_atmosphere.txt",  # packaged file inside empylib/ref_spectra_data
    get_from_local_path=True,
    return_data=True,
)

am15_global = ref.AM15(
    lam,
    spectra_type="global",  # diffuse + direct irradiance
)

am15_direct = ref.AM15(
    lam,
    spectra_type="direct",  # direct normal irradiance
)

t_atm = ref.T_atmosphere(lam)
t_atm_hemi = ref.T_atmosphere_hemi(
    lam,
    beta_tilt=30,  # receiver tilt in degrees
)

b_300 = ref.Bplanck(
    lam,
    T=300,                # blackbody temperature in Kelvin
    unit="wavelength",    # return B_lambda
)

display(atm_table.head())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lam, am15_global / am15_global.max(), label="AM1.5 global")
ax.plot(lam, am15_direct / am15_direct.max(), label="AM1.5 direct")
ax.plot(lam, t_atm, label="Atmospheric transmission")
ax.plot(lam, t_atm_hemi, label="Hemispherical atmosphere")
ax.plot(lam, b_300 / np.nanmax(b_300), label="Planck spectrum at 300 K")
ax.set_xlabel("Wavelength (um)")
ax.set_ylabel("Normalized response")
ax.set_title("Reference spectra used across EMPI Lib")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

**How to read the result**

The solar spectra dominate at short wavelengths, while the 300 K Planck curve lives in the thermal infrared. The atmosphere functions are already dimensionless transmission factors, so they should stay between 0 and 1.

**Common pitfalls**

- Do not mix normalized curves with physically scaled curves when you need an energy balance
- The hemispherical atmosphere depends on tilt, so keep that parameter consistent with your geometry
- For thermal emission you usually want wavelengths extending well into the infrared

**Try this next**

- Change `T` in `Bplanck` to 500 K or 1000 K and watch the peak shift
- Overlay `atm_values` and `t_atm` to confirm the file reader and helper agree

## Compute photopic and spectrum-weighted averages

**Functions used**

- ref.yCIE_lum
- ref.spectral_average

**Problem we are solving**

Once you have a wavelength-dependent property, you often need a single weighted number: visible brightness, solar absorptance, or thermal emittance. This section shows the common workflow.

**Parameter guide for this example**

- `lam_visible`: visible wavelength range for the CIE photopic luminosity function
- `lam_property`: wavelength range of the property you want to average
- `absorptance`: spectral property sampled on `lam_property`
- `spectrum='solar'` or `'thermal'`: chooses the weighting kernel used by `spectral_average`

**Outputs to inspect**

- `y_lum`: photopic sensitivity function
- `solar_average` and `thermal_average`: scalar weighted averages

In [ ]:
lam_visible = np.linspace(0.38, 0.78, 200)
y_lum = ref.yCIE_lum(lam_visible)

lam_property = np.linspace(0.30, 25.0, 400)
absorptance = 0.15 + 0.75 * np.exp(-((lam_property - 1.10) / 0.60) ** 2)

solar_average = ref.spectral_average(
    lam_property,
    absorptance,
    spectrum="solar",  # AM1.5-weighted average
)

thermal_average = ref.spectral_average(
    lam_property,
    absorptance,
    spectrum="thermal", # Planck-weighted average
    T=500,              # emitter temperature in Kelvin
)

print("Solar-weighted average:", solar_average)
print("Thermal-weighted average at 500 K:", thermal_average)

fig, ax = plt.subplots()
ax.plot(lam_visible, y_lum, color="tab:green", label="y_CIE")
ax.set_xlabel("Wavelength (um)")
ax.set_ylabel("Relative response")
ax.set_title("Photopic luminosity function")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

**How to read the result**

The luminosity curve peaks near green wavelengths because that is where the photopic human eye is most sensitive. The two scalar averages differ because solar weighting and 500 K thermal weighting emphasize very different spectral regions.

**Common pitfalls**

- A weighted average is only meaningful if your wavelength grid covers the range where the weighting spectrum is important
- If you use `spectrum='thermal'`, always pass the temperature you actually care about

**Try this next**

- Replace `absorptance` with a measured reflectance or transmittance spectrum
- Compare thermal averages at 300 K, 500 K, and 1000 K

## Use `plot_spectra` for quick publication-style comparisons

**Functions used**

- ref.plot_spectra

**Problem we are solving**

You will often want to show more than one spectral quantity on the same axes with the built-in solar and thermal backgrounds. `plot_spectra` gives you a consistent plotting style for that job.

**Parameter guide for this example**

- each curve is passed as `(lam, values, style_dict)`
- `ylabel`: label for the plotted spectral quantity
- `title`: figure title
- `show_background_legend=True`: include the background guides in the legend

**Outputs to inspect**

- `fig, ax`: the matplotlib handles returned by `plot_spectra`
- a ready-to-edit figure with a consistent reference-spectrum background

In [ ]:
fig, ax = ref.plot_spectra(
    (
        lam_property,
        absorptance,
        {
            "label": "Absorptance",
            "color": "black",
        },
    ),
    (
        lam_property,
        1 - absorptance,
        {
            "label": "Reflectance surrogate",
            "color": "tab:blue",
        },
    ),
    ylabel="Dimensionless spectrum",
    title="Using plot_spectra with standard backgrounds",
    show_background_legend=True,
)
plt.show()

**How to read the result**

This helper is convenient when you want consistent context around a spectrum, especially if your readers need to see where the solar and thermal bands sit relative to your data.

**Common pitfalls**

- The style dictionaries are passed directly to matplotlib, so invalid keys will raise plotting errors
- If you compare spectra sampled on different wavelength grids, keep the units consistent

**Try this next**

- Add a third curve such as transmittance or emissivity
- Customize the line styles and colors in the per-curve dictionaries